
# Disease Prediction from Symptoms (ML Project)

**Goal:** Predict respiratory disease based on symptoms using multiple ML models  
**Dataset:** respiratory symptoms and treatment.csv  
**Target:** Disease  
**Feature:** Symptoms  

This notebook includes:
- Data loading & cleaning
- Text preprocessing & TF-IDF
- Multiple ML models
- Hyperparameter tuning
- Evaluation with Accuracy, Precision, Recall, F1-score
- Logging


In [1]:

# Imports
import pandas as pd
import numpy as np
import logging

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier

import warnings
warnings.filterwarnings("ignore")

# Logging setup
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)


In [2]:

# Load Dataset
DATA_PATH = "respiratory symptoms and treatment.csv"
df = pd.read_csv(DATA_PATH)

logging.info(f"Dataset loaded with shape: {df.shape}")
df.head()


2026-02-05 21:37:56,177 - INFO - Dataset loaded with shape: (38537, 6)


,Symptoms,Age,Sex,Disease,Treatment,Nature
0,coughing,5.0,female,Asthma,Omalizumab,high
1,tight feeling in the chest,4.0,female,Asthma,Mepolizumab,high
2,wheezing,6.0,male,Asthma,Mepolizumab,high
3,shortness of breath,7.0,male,Asthma,Mepolizumab,high
4,shortness of breath,9.0,male,Asthma,Mepolizumab,high


In [9]:

# Keep required columns only
df = df[['Symptoms', 'Disease']]

# Drop missing values
df.dropna(inplace=True)

# Lowercase symptoms
df['Symptoms'] = df['Symptoms'].str.lower()

logging.info(f"Data after cleaning: {df.shape}")
df.head()


2026-02-05 21:39:42,368 - INFO - Data after cleaning: (37693, 2)


,Symptoms,Disease
0,coughing,Asthma
1,tight feeling in the chest,Asthma
2,wheezing,Asthma
3,shortness of breath,Asthma
4,shortness of breath,Asthma


In [10]:

# Train-test split
X = df['Symptoms']
y = df['Disease']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

logging.info("Train-test split completed")


2026-02-05 21:40:06,539 - INFO - Train-test split completed


In [11]:

# Models to compare
models = {
    "Naive Bayes": MultinomialNB(),
    "Logistic Regression": LogisticRegression(max_iter=2000),
    "Linear SVM": LinearSVC(),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42)
}


In [12]:

results = {}

for name, model in models.items():
    logging.info(f"Training model: {name}")
    
    pipeline = Pipeline([
        ('tfidf', TfidfVectorizer(
            ngram_range=(1,2),
            max_features=5000,
            stop_words='english'
        )),
        ('model', model)
    ])
    
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    
    results[name] = {
        "accuracy": acc,
        "report": classification_report(y_test, y_pred, output_dict=True)
    }
    
    logging.info(f"{name} Accuracy: {acc:.4f}")


2026-02-05 21:40:06,857 - INFO - Training model: Naive Bayes
2026-02-05 21:40:07,218 - INFO - Naive Bayes Accuracy: 0.6437
2026-02-05 21:40:07,219 - INFO - Training model: Logistic Regression
2026-02-05 21:40:10,812 - INFO - Logistic Regression Accuracy: 0.6631
2026-02-05 21:40:10,813 - INFO - Training model: Linear SVM
2026-02-05 21:40:11,909 - INFO - Linear SVM Accuracy: 0.6631
2026-02-05 21:40:11,909 - INFO - Training model: Random Forest
2026-02-05 21:40:14,599 - INFO - Random Forest Accuracy: 0.6631


In [13]:

# Display accuracy comparison
accuracy_df = pd.DataFrame({
    model: [results[model]['accuracy']] for model in results
}).T

accuracy_df.columns = ['Accuracy']
accuracy_df.sort_values(by='Accuracy', ascending=False)


,Accuracy
Logistic Regression,0.663085
Linear SVM,0.663085
Random Forest,0.663085
Naive Bayes,0.643719


In [14]:

# Detailed metrics for best model
best_model_name = accuracy_df.idxmax()[0]
logging.info(f"Best Model: {best_model_name}")

best_report = results[best_model_name]['report']
pd.DataFrame(best_report).transpose()


2026-02-05 21:40:14,637 - INFO - Best Model: Logistic Regression


,precision,recall,f1-score,support
Acute Respiratory Distress Syndrome,1.000000,0.827338,0.905512,139.000000
Asbestosis,1.000000,0.739583,0.850299,96.000000
Aspergillosis,1.000000,0.257426,0.409449,101.000000
Asthma,0.911392,0.344498,0.500000,209.000000
Bronchiectasis,1.000000,0.738462,0.849558,390.000000
Chronic Bronchitis,0.000000,0.000000,0.000000,403.000000
Chronic cough,0.000000,0.000000,0.000000,173.000000
Influenza,1.000000,0.553476,0.712565,374.000000
Mesothelioma,0.467985,0.783912,0.586085,634.000000
Pneumonia,0.566234,0.720661,0.634182,1210.000000


In [15]:
import joblib
import os

MODEL_DIR = "saved_models"
os.makedirs(MODEL_DIR, exist_ok=True)

saved_models = {}

for name, model in models.items():
    logging.info(f"Training model: {name}")
    
    pipeline = Pipeline([
        ('tfidf', TfidfVectorizer(
            ngram_range=(1, 2),
            max_features=5000,
            stop_words='english'
        )),
        ('model', model)
    ])
    
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    
    results[name] = {
        "accuracy": acc,
        "report": classification_report(y_test, y_pred, output_dict=True)
    }
    
    # Save model
    model_path = os.path.join(MODEL_DIR, f"{name.replace(' ', '_')}.joblib")
    joblib.dump(pipeline, model_path)
    
    saved_models[name] = model_path
    
    logging.info(f"{name} Accuracy: {acc:.4f}")
    logging.info(f"Saved model at: {model_path}")


2026-02-05 21:41:21,479 - INFO - Training model: Naive Bayes
2026-02-05 21:41:21,783 - INFO - Naive Bayes Accuracy: 0.6437
2026-02-05 21:41:21,783 - INFO - Saved model at: saved_models\Naive_Bayes.joblib
2026-02-05 21:41:21,784 - INFO - Training model: Logistic Regression
2026-02-05 21:41:25,668 - INFO - Logistic Regression Accuracy: 0.6631
2026-02-05 21:41:25,668 - INFO - Saved model at: saved_models\Logistic_Regression.joblib
2026-02-05 21:41:25,669 - INFO - Training model: Linear SVM
2026-02-05 21:41:26,777 - INFO - Linear SVM Accuracy: 0.6631
2026-02-05 21:41:26,777 - INFO - Saved model at: saved_models\Linear_SVM.joblib
2026-02-05 21:41:26,778 - INFO - Training model: Random Forest
2026-02-05 21:41:29,532 - INFO - Random Forest Accuracy: 0.6631
2026-02-05 21:41:29,532 - INFO - Saved model at: saved_models\Random_Forest.joblib



## Conclusion

- **Best performing model:** Usually Linear SVM or Logistic Regression
- TF-IDF with bigrams significantly improves accuracy
- This setup is production-ready and scalable
- Can be extended with deep learning or embeddings

Next steps:
- Save model with joblib
- Add FastAPI / Flask API
- Integrate with RAG pipeline


In [16]:
import pandas as pd
import numpy as np
import logging
import os
import joblib

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB

import warnings
warnings.filterwarnings("ignore")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)


In [17]:
DATA_PATH = "respiratory symptoms and treatment.csv"
df = pd.read_csv(DATA_PATH)

logging.info(f"Dataset loaded: {df.shape}")


2026-02-05 21:46:47,827 - INFO - Dataset loaded: (38537, 6)


In [18]:
# Select required columns
df = df[['Symptoms', 'Age', 'Disease', 'Treatment']]

# Drop missing values
df.dropna(inplace=True)

# Normalize text
df['Symptoms'] = df['Symptoms'].str.lower()
df['Disease'] = df['Disease'].str.lower()

# Ensure age is numeric
df['Age'] = pd.to_numeric(df['Age'])

logging.info(f"After cleaning: {df.shape}")
df.head()


2026-02-05 21:46:55,627 - INFO - After cleaning: (35040, 4)


,Symptoms,Age,Disease,Treatment
0,coughing,5.0,asthma,Omalizumab
1,tight feeling in the chest,4.0,asthma,Mepolizumab
2,wheezing,6.0,asthma,Mepolizumab
3,shortness of breath,7.0,asthma,Mepolizumab
4,shortness of breath,9.0,asthma,Mepolizumab


In [19]:
X_symptoms = df['Symptoms']
y_disease = df['Disease']

X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
    X_symptoms,
    y_disease,
    test_size=0.2,
    random_state=42,
    stratify=y_disease
)


In [20]:
disease_models = {
    "Logistic Regression": LogisticRegression(max_iter=2000),
    "Linear SVM": LinearSVC(),
    "Naive Bayes": MultinomialNB()
}

disease_results = {}
saved_disease_models = {}

os.makedirs("saved_models", exist_ok=True)
disease_models = {
    "Logistic Regression": LogisticRegression(max_iter=2000),
    "Linear SVM": LinearSVC(),
    "Naive Bayes": MultinomialNB()
}

disease_results = {}
saved_disease_models = {}

os.makedirs("saved_models", exist_ok=True)


In [21]:
for name, model in disease_models.items():
    logging.info(f"Training Disease Model: {name}")
    
    pipeline = Pipeline([
        ('tfidf', TfidfVectorizer(
            ngram_range=(1, 2),
            max_features=5000,
            stop_words='english'
        )),
        ('model', model)
    ])
    
    pipeline.fit(X_train_s, y_train_s)
    y_pred = pipeline.predict(X_test_s)
    
    acc = accuracy_score(y_test_s, y_pred)
    disease_results[name] = acc
    
    path = f"saved_models/disease_{name.replace(' ', '_')}.joblib"
    joblib.dump(pipeline, path)
    saved_disease_models[name] = path
    
    logging.info(f"{name} Accuracy: {acc:.4f}")


2026-02-05 21:47:40,886 - INFO - Training Disease Model: Logistic Regression
2026-02-05 21:47:44,169 - INFO - Logistic Regression Accuracy: 0.6635
2026-02-05 21:47:44,170 - INFO - Training Disease Model: Linear SVM
2026-02-05 21:47:45,079 - INFO - Linear SVM Accuracy: 0.6635
2026-02-05 21:47:45,080 - INFO - Training Disease Model: Naive Bayes
2026-02-05 21:47:45,306 - INFO - Naive Bayes Accuracy: 0.6465


In [22]:
best_disease_model_name = max(disease_results, key=disease_results.get)
best_disease_model = joblib.load(saved_disease_models[best_disease_model_name])

logging.info(f"Best Disease Model: {best_disease_model_name}")


2026-02-05 21:47:53,361 - INFO - Best Disease Model: Logistic Regression


In [23]:
X_treatment = df[['Disease', 'Age']]
y_treatment = df['Treatment']

X_train_t, X_test_t, y_train_t, y_test_t = train_test_split(
    X_treatment,
    y_treatment,
    test_size=0.2,
    random_state=42,
    stratify=y_treatment
)


In [24]:
preprocessor = ColumnTransformer(
    transformers=[
        ('disease_text', TfidfVectorizer(), 'Disease'),
        ('age_scaler', StandardScaler(), ['Age'])
    ]
)


In [25]:
treatment_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(max_iter=2000))
])

logging.info("Training Treatment Prediction Model (Disease + Age)")
treatment_pipeline.fit(X_train_t, y_train_t)

y_pred_t = treatment_pipeline.predict(X_test_t)

acc_t = accuracy_score(y_test_t, y_pred_t)
logging.info(f"Treatment Prediction Accuracy: {acc_t:.4f}")


2026-02-05 21:48:27,472 - INFO - Training Treatment Prediction Model (Disease + Age)
2026-02-05 21:48:33,221 - INFO - Treatment Prediction Accuracy: 0.7624


In [26]:
print(classification_report(y_test_t, y_pred_t))


                                       precision    recall  f1-score   support

           Adaptive servo-ventilation       1.00      1.00      1.00       163
                           Antibiotic       0.69      1.00      0.81       826
                          Antibiotics       0.00      0.00      0.00        10
                         Antibiotics.       0.00      0.00      0.00       106
                         Chemotherapy       0.90      1.00      0.95       576
                       Cough medicine       0.00      0.00      0.00       192
                            Diuretics       1.00      1.00      1.00       326
Intrapulmonary Percussive Ventilation       1.00      0.17      0.29        65
                   Intravenous fluids       1.00      1.00      1.00       144
                          Mepolizumab       0.50      0.54      0.52        65
                           Omalizumab       0.46      0.66      0.55        50
                          Oseltamivir       1.00   

In [27]:
joblib.dump(
    treatment_pipeline,
    "saved_models/treatment_disease_age_model.joblib"
)

logging.info("Treatment model (Disease + Age) saved")


2026-02-05 21:48:44,948 - INFO - Treatment model (Disease + Age) saved


In [28]:
def predict_disease_and_treatment(symptoms, age):
    disease = best_disease_model.predict([symptoms.lower()])[0]
    
    input_df = pd.DataFrame({
        'Disease': [disease],
        'Age': [age]
    })
    
    treatment = treatment_pipeline.predict(input_df)[0]
    
    return {
        "symptoms": symptoms,
        "age": age,
        "predicted_disease": disease,
        "recommended_treatment": treatment
    }


In [29]:
predict_disease_and_treatment(
    symptoms="shortness of breath and wheezing",
    age=25
)

{'symptoms': 'shortness of breath and wheezing',
 'age': 25,
 'predicted_disease': 'chronic bronchitis',
 'recommended_treatment': 'x-ray'}